Hi! I am a Kaggle beginner and not fluent in English, so this notebook is written in Japanese.

Please use your browser's translation (or DeepL/ChatGPT) to read it!

Hope it helps!

# データの読み込み・ライブラリのインポート

In [1]:
import pandas as pd
import numpy as np

import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

import kagglehub
import os

In [2]:
path = kagglehub.competition_download("playground-series-s6e11")

train = pd.read_csv(os.path.join(path, "train.csv"))
test = pd.read_csv(os.path.join(path, "test.csv"))
sample_submission = pd.read_csv(os.path.join(path, "sample_submission.csv"))

# 関数の定義

In [3]:
def train_lgb(train_df, test_df, feature_cols, target_col='is_fraud'):
    """
    指定された特徴量で LightGBM の交差検証学習を行い、
    OOF予測値（確率）、テスト予測値（確率）、学習済みモデル群を返す関数
    """
    
    # bool型をそのまま int（1/0）に変換
    y_train = train_df[target_col].astype(int).values
    
    # ループの前にカテゴリ型への一括変換
    X_train_df = train_df[feature_cols].copy()
    X_test_df = test_df[feature_cols].copy()
    
    for col in feature_cols:
        if X_train_df[col].dtype == 'object':
            X_train_df[col] = X_train_df[col].astype('category')
            X_test_df[col] = X_test_df[col].astype('category')

    X_train_df = X_train_df.reset_index(drop=True)

    cv = KFold(n_splits=5, shuffle=True, random_state=0)
    
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',

        'num_leaves': 4,
        
        'n_jobs': -1,
        'random_state': 0,
        'verbose': -1,
        'enable_categorical': True,
    }

    models = []
    oof_train = np.zeros(len(X_train_df))
    
    for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train_df, y_train)):
        X_tr = X_train_df.iloc[train_idx]
        y_tr = y_train[train_idx]
        X_va = X_train_df.iloc[valid_idx]
        y_va = y_train[valid_idx]

        # Datasetの作成 (free_raw_data=True でメモリを節約)
        lgb_train = lgb.Dataset(X_tr, y_tr, free_raw_data=True)
        lgb_eval = lgb.Dataset(X_va, y_va, reference=lgb_train, free_raw_data=True)

        model = lgb.train(
            params,
            lgb_train,
            valid_sets=[lgb_eval],
            num_boost_round=1000,
            callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
        )

        best_iter = model.best_iteration
        best_score = model.best_score['valid_0']['auc']
        print(f'Fold {fold+1} | Best Iteration: {best_iter:4d} | Best AUC: {best_score:.5f}')

        oof_train[valid_idx] = model.predict(X_va, num_iteration=model.best_iteration)
        
        models.append(model)

    # 全体の OOF ROC-AUC スコアを計算
    score = roc_auc_score(y_train, oof_train)
    print(f'\nOOF ROC-AUC Score: {score:.5f}')

    # すべての学習が終わった後、ループの外でテストデータを一括予測する
    y_preds = [model.predict(X_test_df, num_iteration=model.best_iteration) for model in models]
    y_preds_mean = np.mean(y_preds, axis=0)

    return oof_train, y_preds_mean, models

# 実行

In [4]:
%%time

feature_cols = [
    'amount_usd',
    'hours_since_last_txn',
    'txn_count_last_24h',  
    'distance_from_home_km',
    'card_age_months', 
    'customer_age',
    'account_balance_usd',
    'cvv_retry_count', 
    'velocity_score',
    'time_of_day_hour',
    'day_of_week', 
    'merchant_risk_score',
    'prior_disputes',
    'merchant_category',
    'card_type',
    'auth_method', 
    'channel',
    'device_type',
    'is_foreign_transaction',  
    'is_new_merchant',  
    'used_vpn',  
    'ip_country_mismatch',
    'billing_shipping_mismatch', 
    'is_ai_generated_scam_attempt'  
]


# 関数の実行
oof, preds, models = train_lgb(train, test, feature_cols)


# 特徴量重要度の可視化（表形式で出力）
feature_importances = np.mean(
    [model.feature_importance(importance_type='gain') for model in models],
    axis=0,
)

importance_df = (
    pd.DataFrame({'feature': feature_cols, 'importance': feature_importances})
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)

display(importance_df.head(20))

Fold 1 | Best Iteration:  678 | Best AUC: 0.94471
Fold 2 | Best Iteration:  935 | Best AUC: 0.95136
Fold 3 | Best Iteration:  983 | Best AUC: 0.94363
Fold 4 | Best Iteration:  375 | Best AUC: 0.94572
Fold 5 | Best Iteration:  541 | Best AUC: 0.94839

OOF ROC-AUC Score: 0.94674


,feature,importance
0,cvv_retry_count,48111.981146
1,merchant_risk_score,44879.403119
2,auth_method,29784.205487
3,velocity_score,28075.082724
4,ip_country_mismatch,26863.718465
5,is_new_merchant,18440.077794
6,customer_age,10986.272694
7,billing_shipping_mismatch,10434.614691
8,used_vpn,7353.546231
9,is_foreign_transaction,6235.850647


CPU times: user 8min 32s, sys: 1.02 s, total: 8min 33s
Wall time: 2min 11s


# 提出

In [5]:
submission = sample_submission.copy()
submission['is_fraud'] = preds

submission.to_csv("submission.csv", index=False)
print("submission.csv を作成しました！")


# ファイルの内容・件数・欠損値のチェック
print('\n--- 先頭 5 行の確認 ---')
print(submission.head())

print('\n--- データ件数の確認 ---')
print(f'作成した件数: {len(submission)} / 元の件数: {len(sample_submission)}')

print('\n--- 欠損値（Null）の有無チェック ---')
print(f'欠損値の数: {submission['is_fraud'].isnull().sum()}')

submission.csv を作成しました！

--- 先頭 5 行の確認 ---
       id  is_fraud
0  699960  0.000277
1  699961  0.000212
2  699962  0.083304
3  699963  0.001577
4  699964  0.001273

--- データ件数の確認 ---
作成した件数: 299984 / 元の件数: 299984

--- 欠損値（Null）の有無チェック ---
欠損値の数: 0
